In [ ]:
import numpy as np
import openpmd_viewer as ioview
%matplotlib widget
import scipy.constants as sc
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from dataclasses import dataclass

import re
import shutil
import subprocess
from pathlib import Path
import json
import openpmd_api as io

In [ ]:
ts = ioview.OpenPMD

In [ ]:
def run_bpls(bp_path, args=("-l",), timeout=30):
    """
    Run ADIOS2 bpls on a BP file/directory and return stdout.

    Examples:
      run_bpls("diags/diag1/openpmd.bp")
      run_bpls("diags/diag1/openpmd.bp", args=("-la",))
    """
    if shutil.which("bpls") is None:
        raise RuntimeError("Could not find `bpls` in PATH.")

    cmd = ["bpls", *args, str(bp_path)]
    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
        timeout=timeout,
    )

    if result.returncode != 0:
        print("STDERR:")
        print(result.stderr)
        raise RuntimeError(f"`{' '.join(cmd)}` failed")

    return result.stdout

In [ ]:
def _read_linear_access():
    for enum_name in ("Access_Type", "Access"):
        enum = getattr(io, enum_name, None)
        if enum is not None and hasattr(enum, "read_linear"):
            return enum.read_linear
    raise RuntimeError("Could not find openPMD read_linear access enum.")


def _resolve_mesh_component(iteration, field):
    """
    field examples:
      "rho"
      "E/x"
      "E/z"
      "B/y"
    """
    if "/" in field:
        mesh_name, component = field.split("/", 1)
    else:
        mesh_name, component = field, None

    mesh = iteration.meshes[mesh_name]

    if component is not None:
        rc = mesh[component]
        return mesh, rc, mesh_name, component

    # Scalar record component case
    if hasattr(mesh, "load_chunk") and hasattr(mesh, "shape"):
        return mesh, mesh, mesh_name, None

    comps = list(mesh.keys())
    if len(comps) == 1:
        component = comps[0]
        rc = mesh[component]
        return mesh, rc, mesh_name, component

    raise ValueError(
        f"Field {mesh_name!r} has components {comps}. "
        f"Use e.g. {mesh_name}/x, {mesh_name}/y, or {mesh_name}/z."
    )

In [ ]:
def _meta_per_dim(meta, key):
    """
    Return a tuple of length ndim even if meta[key] is scalar.
    """
    ndim = len(meta["axis_labels"])
    val = meta[key]

    if np.isscalar(val):
        return (float(val),) * ndim

    val = tuple(val)
    if len(val) != ndim:
        raise ValueError(f"meta[{key!r}] has length {len(val)}, expected {ndim}")
    return val

In [ ]:
def axis_centers(meta, dim):
    """
    Physical coordinates of cell centers along one dimension.
    """
    spacing = _meta_per_dim(meta, "grid_spacing")[dim]
    global_offset = _meta_per_dim(meta, "grid_global_offset")[dim]
    grid_unit_SI = _meta_per_dim(meta, "grid_unit_SI")[dim]
    position = _meta_per_dim(meta, "position")[dim]

    i0 = meta["offset"][dim]
    n = meta["extent"][dim]

    centers = global_offset + (i0 + np.arange(n) + position) * spacing
    return centers * grid_unit_SI

In [ ]:
def axis_edges(meta, dim):
    """
    Physical coordinates of cell edges along one dimension.
    Good for pcolormesh.
    """
    c = axis_centers(meta, dim)

    if len(c) == 1:
        spacing = _meta_per_dim(meta, "grid_spacing")[dim]
        grid_unit_SI = _meta_per_dim(meta, "grid_unit_SI")[dim]
        dx = spacing * grid_unit_SI
        return np.array([c[0] - 0.5 * dx, c[0] + 0.5 * dx])

    dx = np.diff(c)
    return np.concatenate((
        [c[0] - 0.5 * dx[0]],
        0.5 * (c[:-1] + c[1:]),
        [c[-1] + 0.5 * dx[-1]],
    ))

In [ ]:
def reduce_warpx_slice(arr, meta, slice_axis=0, slice_reduction="mean"):
    """
    Reduce a 3D WarpX slice output (often 2 cells thick in one direction)
    to a 2D image.

    slice_reduction:
      - "mean"  : average across slice_axis
      - "first" : take index 0 along slice_axis
    """
    arr = np.asarray(arr)

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {arr.shape}")

    if slice_reduction == "mean":
        img = arr.mean(axis=slice_axis)
        slice_note = f"avg over {meta['extent'][slice_axis]} cells"
        slice_coord = np.mean(axis_centers(meta, slice_axis))

    elif slice_reduction == "first":
        slicer = [slice(None)] * 3
        slicer[slice_axis] = 0
        img = arr[tuple(slicer)]
        slice_note = "cell 0"
        slice_coord = axis_centers(meta, slice_axis)[0]

    else:
        raise ValueError("slice_reduction must be 'mean' or 'first'")

    remaining_dims = [d for d in range(3) if d != slice_axis]
    ydim, xdim = remaining_dims   # img.shape == (len(ydim), len(xdim))

    return img, xdim, ydim, slice_coord, slice_note

In [ ]:
series_path = "../diags/fields_sliced/openpmd.bp5"   # <-- change this

txt = run_bpls(series_path, args=("-l",))
print(txt[:5000])

In [ ]:
def read_field_linear_capped(
    series_path,
    target_iteration,
    field,
    *,
    max_linear_steps,
    offset=None,
    extent=None,
    verbose=True,
):
    """
    Read one field from one iteration using openPMD-api linear mode.

    Crucially, this stops after max_linear_steps even if the ADIOS2 stream
    never reaches a clean EOF.

    Parameters
    ----------
    series_path : str
        Path to openPMD ADIOS2 BP output.
    target_iteration : int
        Iteration to read.
    field : str
        Examples: "rho", "E/x", "B/z".
    max_linear_steps : int
        Hard cap on how many linear iterations to inspect.
    offset, extent : list[int] or None
        Optional chunk selection.
    """
    series = io.Series(str(series_path), _read_linear_access())

    try:
        for n, it in enumerate(series.read_iterations()):
            if n >= max_linear_steps:
                raise RuntimeError(
                    f"Stopped after max_linear_steps={max_linear_steps} "
                    f"without finding iteration {target_iteration}."
                )

            idx = int(it.iteration_index)

            if verbose:
                print(f"step {n}: iteration {idx}")

            if idx != target_iteration:
                it.close()
                continue

            mesh, rc, mesh_name, component = _resolve_mesh_component(it, field)
            resolved_name = field

            shape = list(rc.shape)
            ndim = len(shape)

            if offset is None:
                offset_use = [0] * ndim
            else:
                offset_use = list(offset)

            if extent is None:
                extent_use = shape
            else:
                extent_use = list(extent)

            if verbose:
                print(f"Reading {resolved_name} at iteration {idx}")
                print(f"shape  = {shape}")
                print(f"offset = {offset_use}")
                print(f"extent = {extent_use}")

            meta = {
                "iteration": int(it.iteration_index),
                "field": field,
                "mesh_name": mesh_name,
                "component": component,
                "shape": tuple(rc.shape),
                "offset": tuple(offset_use),
                "extent": tuple(extent_use),
                "axis_labels": tuple(mesh.axis_labels),
                "grid_spacing": tuple(mesh.grid_spacing),
                "grid_global_offset": tuple(mesh.grid_global_offset),
                "grid_unit_SI": mesh.grid_unit_SI,
                "position": tuple(rc.position),
                "unit_SI": rc.unit_SI,
                "time": float(it.time),
                "time_unit_SI": float(it.time_unit_SI),
            }
            
            data_proxy = rc.load_chunk(offset_use, extent_use)
            series.flush()
            arr = np.array(data_proxy, copy=True)
            
            it.close()
            
            return arr, meta

        raise RuntimeError(
            f"read_iterations() ended before finding iteration {target_iteration}."
        )

    finally:
        series.close()

In [ ]:
target_iteration = 14928
field = "E/z"

arr, meta = read_field_linear_capped(
    series_path,
    target_iteration,
    field,
    max_linear_steps=1245,   # manually chosen safety cap,
    verbose=False
)

arr.shape

In [ ]:
meta

## Plot Data

In [ ]:
def plot_warpx_slice(
    arr,
    meta,
    *,
    slice_axis=0,
    slice_reduction="mean",
    absmax=None,
    autoscale_percentile=None,
    cmap="RdBu_r",
    colorbar_unit="V/m",
    fig=None,
    ax=None,
    cax=None,
):
    """
    Plot a WarpX slice with fixed layout and per-frame symmetric color scaling.

    If absmax is None:
      - use per-frame max(abs(img))
    If autoscale_percentile is set, e.g. 99.5:
      - use percentile(abs(img), 99.5), still symmetric
    """
    img, xdim, ydim, slice_coord, slice_note = reduce_warpx_slice(
        arr,
        meta,
        slice_axis=slice_axis,
        slice_reduction=slice_reduction,
    )

    x_edges = axis_edges(meta, xdim)
    y_edges = axis_edges(meta, ydim)

    if absmax is None:
        finite_abs = np.abs(img[np.isfinite(img)])

        if finite_abs.size == 0:
            absmax = 1.0
        elif autoscale_percentile is None:
            absmax = np.max(finite_abs)
        else:
            absmax = np.percentile(finite_abs, autoscale_percentile)

    if absmax == 0:
        absmax = 1.0

    vmin = -absmax
    vmax = +absmax

    # Fixed layout for movie consistency.
    # These positions do not depend on tick labels or colorbar label size.
    if fig is None:
        fig = plt.figure(figsize=(8.0, 6.0), dpi=150)

    if ax is None:
        ax = fig.add_axes([0.10, 0.10, 0.68, 0.78])

    if cax is None:
        cax = fig.add_axes([0.82, 0.12, 0.03, 0.74])

    im = ax.pcolormesh(
        x_edges,
        y_edges,
        img,
        shading="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        rasterized=True,
    )

    cb = fig.colorbar(im, cax=cax)

    cbar_label = meta["field"]
    if colorbar_unit:
        cbar_label += f" [{colorbar_unit}]"
    cb.set_label(cbar_label)

    xlab = meta["axis_labels"][xdim]
    ylab = meta["axis_labels"][ydim]
    ax.set_xlabel(f"{xlab} [m]")
    ax.set_ylabel(f"{ylab} [m]")

    t_si = meta.get("time", 0.0) * meta.get("time_unit_SI", 1.0)
    slice_axis_label = meta["axis_labels"][slice_axis]

    ax.set_title(
        f"{meta['field']}   it={meta['iteration']}   t={t_si:.6e} s\n"
        f"{slice_axis_label}={slice_coord:.6e} m   "
        f"{slice_note}   range=[{vmin:.3e}, {vmax:.3e}]"
    )
    ax.set_aspect(1.0)

    return fig, ax, cax, im

In [ ]:
def copy_meta_from_iteration(it, mesh, rc, field, mesh_name, component, offset_use, extent_use):
    return {
        "iteration": int(it.iteration_index),
        "field": field,
        "mesh_name": mesh_name,
        "component": component,
        "shape": tuple(rc.shape),
        "offset": tuple(offset_use),
        "extent": tuple(extent_use),
        "axis_labels": tuple(mesh.axis_labels),
        "grid_spacing": tuple(mesh.grid_spacing),
        "grid_global_offset": tuple(mesh.grid_global_offset),
        "grid_unit_SI": mesh.grid_unit_SI,
        "position": tuple(rc.position),
        "unit_SI": rc.unit_SI,
        "time": float(it.time),
        "time_unit_SI": float(it.time_unit_SI),
    }

In [ ]:
def save_series_frames_linear(
    series_path,
    field,
    outdir,
    *,
    max_linear_steps,
    offset=None,
    extent=None,
    every=1,
    slice_axis=0,
    slice_reduction="mean",
    absmax=None,
    autoscale_percentile=None,
    cmap="RdBu_r",
    colorbar_unit="V/m",
    dpi=150,
    prefix="frame",
    verbose=True,
):
    """
    Save PNG frames with:
      - fixed subplot positions
      - per-frame symmetric colorbar limits
      - optional robust per-frame autoscaling via autoscale_percentile
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    saved = []

    frame_no = 0
    for arr, meta in iter_field_frames_linear(
        series_path,
        field,
        max_linear_steps=max_linear_steps,
        offset=offset,
        extent=extent,
        every=every,
        verbose=verbose,
    ):
        fig = plt.figure(figsize=(8.0, 6.0), dpi=dpi)

        # Fixed axes positions for movie stability.
        ax = fig.add_axes([0.10, 0.10, 0.68, 0.78])
        cax = fig.add_axes([0.82, 0.12, 0.03, 0.74])

        plot_warpx_slice(
            arr,
            meta,
            slice_axis=slice_axis,
            slice_reduction=slice_reduction,
            absmax=absmax,
            autoscale_percentile=autoscale_percentile,
            cmap=cmap,
            colorbar_unit=colorbar_unit,
            fig=fig,
            ax=ax,
            cax=cax,
        )

        fname = outdir / f"{prefix}_{frame_no:06d}_it{meta['iteration']:08d}.png"
        fig.savefig(fname, dpi=dpi)
        plt.close(fig)

        saved.append(fname)
        frame_no += 1

    return saved

In [ ]:
def iter_field_frames_linear(
    series_path,
    field,
    *,
    max_linear_steps,
    offset=None,
    extent=None,
    every=1,
    verbose=True,
):
    """
    Yield (arr, meta) for each iteration in a linear pass.

    Important:
      Stops after max_linear_steps to avoid ever reading indefinitely
      into a broken tail.
    """
    series = io.Series(str(series_path), _read_linear_access())

    try:
        inspected = 0

        for it in series.read_iterations():
            inspected += 1
            idx = int(it.iteration_index)

            try:
                if verbose:
                    print(f"step {inspected - 1}: iteration {idx}")

                if (inspected - 1) % every != 0:
                    continue

                mesh, rc, mesh_name, component = _resolve_mesh_component(it, field)

                full_shape = tuple(int(x) for x in rc.shape)
                ndim = len(full_shape)

                if offset is None:
                    offset_use = tuple(0 for _ in range(ndim))
                else:
                    offset_use = tuple(int(x) for x in offset)

                if extent is None:
                    extent_use = full_shape
                else:
                    extent_use = tuple(int(x) for x in extent)

                data_proxy = rc.load_chunk(offset_use, extent_use)
                series.flush()

                arr = np.array(data_proxy, copy=True)
                meta = copy_meta_from_iteration(
                    it, mesh, rc, field, mesh_name, component, offset_use, extent_use
                )

            finally:
                it.close()

            yield arr, meta

            # Important: stop before requesting a new step
            if inspected >= max_linear_steps:
                break

    finally:
        series.close()

In [ ]:
fig = plt.figure(figsize=(8.0, 6.0), dpi=150)
ax = fig.add_axes([0.10, 0.10, 0.68, 0.78])
cax = fig.add_axes([0.82, 0.12, 0.03, 0.74])

plot_warpx_slice(
    arr,
    meta,
    slice_axis=0,
    slice_reduction="mean",
    absmax=None,
    autoscale_percentile=99.5,
    cmap="RdBu_r",
    colorbar_unit="V/m",
    fig=fig,
    ax=ax,
    cax=cax,
)

plt.show()

In [ ]:
saved = save_series_frames_linear(
    series_path,
    field="E/z",
    outdir="movie_frames_Ez",
    max_linear_steps=1245,
    every=1,
    slice_axis=0,
    slice_reduction="mean",
    absmax=None,
    autoscale_percentile=99.5,  # still gives [-a, +a] per frame
    cmap="RdBu_r",
    colorbar_unit="V/m",
    dpi=150,
    prefix="Ez",
    verbose=False,
)

In [ ]:
import subprocess

cmd = [
    "ffmpeg",
    "-framerate", "30",
    "-pattern_type", "glob",
    "-i", "movie_frames_Ez/Ez_*.png",
    "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2,format=yuv420p",
    "-c:v", "libx264",
    "-preset", "slow",
    "-crf", "18",
    "Ez_movie.mp4",
]

subprocess.run(cmd, check=True)